## Inverting BigGAN using Optimization
This notebook inverts BigGAN using an optimizer. The trained BigGAN models are from [Huggingface](https://huggingface.co/). 

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
import os
import subprocess
import zipfile
import glob
from PIL import Image

In [ ]:
## PyTorch
import torch
import torch.nn as nn
import torch.utils.data as data
from torch.optim import Adam
# Torchvision
import torchvision
from torchvision import transforms
from torchvision.utils import make_grid, save_image
import torchvision.transforms.functional as F
# tqdm
from tqdm.notebook import tqdm, trange
from torchsummary import summary
# lpips
import lpips

In [ ]:
# Configure environment
torch.backends.cudnn.benchmark = True

device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
print("Device:", device)

In [ ]:
def show_tensor_images(image_tensor, 
                       num_images=16, 
                       nrow=4,
                       filename=""):
    image_grid = make_grid(image_tensor[:num_images].detach().cpu(), nrow=nrow)
    img = image_grid.permute(1, 2, 0).squeeze()
    plt.axis('off')
    plt.imshow(img)
    if len(filename) > 0:
        plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()

### BigGAN

In [ ]:
from pytorch_pretrained_biggan import (BigGAN, one_hot_from_names, 
                                       truncated_noise_sample)

Load the pre-trained model

In [ ]:
model = BigGAN.from_pretrained('biggan-deep-256').eval()

In [ ]:
import nltk
nltk.download('wordnet')

Prepare an example input

In [ ]:
truncation = 0.4
class_vector = one_hot_from_names(['dog', 'coffee', 'flower'], batch_size=3)
noise_vector = truncated_noise_sample(truncation=truncation, batch_size=3)

In [ ]:
noise_vector = torch.from_numpy(noise_vector)
class_vector = torch.from_numpy(class_vector)

In [ ]:
noise_vector = noise_vector.to(device)
class_vector = class_vector.to(device)
model.to(device)

Generate an image and then rescale to range [0, 1]

In [ ]:
output = model(noise_vector, class_vector, truncation)

In [ ]:
image_ten = (output + 1) / 2

Display

In [ ]:
plt.figure(figsize=(12,12))
three_filename = 'BigGAN3images.png'
show_tensor_images(image_ten, num_images=3, nrow=3, filename=three_filename)

### Encoder

In [ ]:
class Encoder(nn.Module):    
    def __init__(self):
        super(Encoder, self).__init__()
        self.input = nn.Sequential(
            nn.Conv2d(3, 32, 3, 2, 1),
            nn.LeakyReLU(negative_slope=0.2)
        )
        self.down1 = self.downsample_block(32, 64)
        self.down2 = self.downsample_block(64, 128)
        self.down3 = self.downsample_block(128, 128)
        self.down4 = self.downsample_block(128, 128)
        self.down5 = self.downsample_block(128, 128)
        self.down6 = self.downsample_block(128, 128)
        self.output = nn.Conv2d(128, 128, 4, 2, 1)
        
    def downsample_block(self, in_channels, out_channels, stride = 2):
        block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, stride, 1),
            nn.InstanceNorm2d(out_channels),
            nn.LeakyReLU(negative_slope=0.2)
        )
        return block
        
    def forward(self, image):
        out = self.input(image)
        out = self.down1(out)
        out = self.down2(out)
        out = self.down3(out)
        out = self.down4(out)
        out = self.down5(out)
        out = self.down6(out)
        return self.output(out).reshape(out.shape[0], out.shape[1])

In [ ]:
enc = Encoder().to(device) 

In [ ]:
summary(enc, input_size = (3, 256, 256), batch_size=32)

In [ ]:
outz = enc(output[0:1,:,:,:].detach())

In [ ]:
n_iter = 50000
lr = 0.002
beta_1 = 0.5
beta_2 = 0.999
batch_size = 8
enc_truncation = 1
epoch_show = 5000

In [ ]:
enc_class_vector = one_hot_from_names(['dog'], batch_size=batch_size)

In [ ]:
enc_class_vector = torch.from_numpy(enc_class_vector).to(device)

In [ ]:
enc_opt = Adam(enc.parameters(), lr=lr, betas=(beta_1, beta_2))
enc_loss = torch.nn.MSELoss()

In [ ]:
loss_lst = list()
tqdm_epoch = trange(n_iter)
for epoch in tqdm_epoch:
    avg_loss = 0.0

    # generate latent vectors
    enc_noise_vector = truncated_noise_sample(truncation=enc_truncation, batch_size=batch_size)
    enc_noise_vector = torch.tensor(enc_noise_vector, requires_grad=True).to(device)
    
    # generate images
    sample_imgs = model(enc_noise_vector, enc_class_vector, enc_truncation)

    #  Train Encoder
    enc_opt.zero_grad()
    inv_z = enc(sample_imgs)
    eloss = enc_loss(enc_noise_vector, inv_z)   
    eloss.backward()
    enc_opt.step()
        
    loss_lst.append(eloss)
    tqdm_epoch.set_description('Loss: {:5f}'.format(eloss))
    if (epoch + 1) % epoch_show == 0:
        torch.save(enc.state_dict(), 'invBigGAN_genckpt' + str(epoch) + '.pth')

In [ ]:
iterations = range(1, n_iter + 1)

In [ ]:
loss_lst_cpu = [i.item() for i in loss_lst]

In [ ]:
fig, ax = plt.subplots(figsize=(7,5))

from cycler import cycler
monochrome = (cycler('color', ['k']) * cycler('linestyle', ['-', '--', ':', '-.']))

ax.set_prop_cycle(monochrome)
ax.plot(iterations, loss_lst_cpu, label='Loss')
    
ax.set_xlabel('Iterations')
ax.set_ylabel('Loss')
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
fig.savefig('invBigGAN.png', dpi=300, bbox_inches='tight')

In [ ]:
outz = enc(output[0:1,:,:,:].detach())

In [ ]:
outz.shape

In [ ]:
out_img = model(outz, enc_class_vector[0:1, :], enc_truncation)

In [ ]:
sout_img = (out_img + 1) / 2

In [ ]:
enc_filename = 'invBigGANenc.png'
show_tensor_images(sout_img, num_images=1, nrow=1, filename=enc_filename)

### Inversion

In [ ]:
def find_z(img, generator, img_class, truncation, device, lr=0.001, num_iter=5000):
    
    dist = torch.nn.L1Loss()
    loss_fn_vgg = lpips.LPIPS(net='vgg').to(device)
    min_loss = 10
    beta = 10
    
    min_z = enc(img).detach().clone().requires_grad_(True)
    optZ = torch.optim.Adam([min_z], lr=lr) 
    
    print("Beginning Iteration.")
    tqdm_it = trange(num_iter)
    for i in tqdm_it:
        def closure():
            optZ.zero_grad()
            img_pred = generator(min_z, img_class, truncation)
            loss_d = dist(img, img_pred)
            loss_p = loss_fn_vgg(img, img_pred)
            loss = loss_d + beta * loss_p
            loss.backward()
            return loss
        optZ.step(closure)
    
        if (i + 1) % 100 == 0:
            img_pred = generator(min_z, img_class, truncation)
            loss_d = dist(img, img_pred)
            loss_p = loss_fn_vgg(img, img_pred)
            loss = loss_d + beta * loss_p
            print("[Iter {}] error: {}".format(i+1, loss.item()))
            comb = torch.cat((img, img_pred), 0)
            filename = 'invBigGAN_pair_' + str(i+1) + '.png'
            show_tensor_images((comb+1)/2, 2, 2, filename=filename)

    return min_z

In [ ]:
inv_class_vector = one_hot_from_names(['dog'], batch_size=1)

In [ ]:
inv_class_vector = torch.from_numpy(inv_class_vector)
inv_class_vector = inv_class_vector.to(device)

In [ ]:
filename = 'ChouChou.JPG'

In [ ]:
img = Image.open(filename)

In [ ]:
transform = transforms.Compose([
    transforms.PILToTensor()
])

In [ ]:
image_tensor = transform(img)

In [ ]:
image_tensor = image_tensor / 255

In [ ]:
image_tensor = image_tensor * 2 - 1

In [ ]:
image_tensor = torch.unsqueeze(image_tensor, 0).to(device)

In [ ]:
z_pred = find_z(output[0:1,:,:,:].detach(), model, inv_class_vector, 1, device, lr=0.001, num_iter=5000)